<a href="https://colab.research.google.com/github/sbooeshaghi/seqcheck/blob/main/examples/seqcheck_devel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/IGVF/seqspec.git

Cloning into 'seqspec'...
remote: Enumerating objects: 1775, done.
remote: Counting objects: 100% (472/472), done.
remote: Compressing objects: 100% (179/179), done.
remote: Total 1775 (delta 332), reused 412 (delta 289), pack-reused 1303
Receiving objects: 100% (1775/1775), 1000.69 MiB | 34.11 MiB/s, done.
Resolving deltas: 100% (1153/1153), done.
Updating files: 100% (242/242), done.


In [2]:
!cd seqspec && pip install .

Processing /content/seqspec
  Preparing metadata (setup.py) ... done
  Created wheel for seqspec: filename=seqspec-0.0.0-py3-none-any.whl size=23452 sha256=b329d4c60bf265ccf70f171c9e6914363813e010512c741dca8e54a3a8ab8903
  Stored in directory: /tmp/pip-ephem-wheel-cache-hb_1lid6/wheels/b5/4f/e5/5d32b26a7d311c33cd532645d92dce50e3a9fde39dd0904105
Successfully built seqspec


In [9]:
from seqspec.utils import load_spec
from seqspec.seqspec_find import run_find

In [15]:
# start with simple python implementation of seqcheck
# input
modality = "rna"
fastqs = ["seqspec/specs/dogmaseq-dig/fastqs/rna_R1_SRR18677638.fastq.gz",
          "seqspec/specs/dogmaseq-dig/fastqs/rna_R2_SRR18677638.fastq.gz"]
spec_fn = "seqspec/specs/dogmaseq-dig/spec.yaml"

# read in spec
spec = load_spec(spec_fn)

In [34]:
import os
import gzip
import numpy as np

In [11]:
# get atomic regions associated with each set of fastqs
run_find(spec, modality, os.path.basename(fastqs[0]))

[{'region_id': 'rna_R1_SRR18677638.fastq.gz', 'region_type': 'fastq', 'name': 'Read 1 fastq', 'sequence_type': 'joined', 'onlist': None, 'sequence': 'NNNNNNNNNNNNNNNNNNNNNNNNNNNN', 'min_len': 28, 'max_len': 28, 'regions': [{'region_id': 'rna_cell_bc', 'region_type': 'barcode', 'name': 'Cell Barcode', 'sequence_type': 'onlist', 'onlist': {'filename': 'RNA-737K-arc-v1.txt', 'md5': 'a88cd21e801ae6f9a7d9a48b67ccf693'}, 'sequence': 'NNNNNNNNNNNNNNNN', 'min_len': 16, 'max_len': 16, 'regions': None}, {'region_id': 'rna_umi', 'region_type': 'umi', 'name': 'umi', 'sequence_type': 'random', 'onlist': None, 'sequence': 'NNNNNNNNNNNN', 'min_len': 12, 'max_len': 12, 'regions': None}]}]

In [47]:
# given a fastq, go through the first 1000 reads and make sure they are the same length as the stated length of the fastq region
r = fastqs[0]
min_expected = run_find(spec, modality, os.path.basename(fastqs[0]))[0].min_len
max_expected = run_find(spec, modality, os.path.basename(fastqs[0]))[0].max_len
lns = []
minl = 9999
maxl = 0
with gzip.open(r, 'r') as f:
  for lidx, l in enumerate(f, 1):
    if lidx%4000==0:
      break
    if (lidx-2%4) == 0:
      ln = len(l.strip())

      if ln < minl:
        minl = ln
      if ln > maxl:
        maxl = ln

      lns.append(ln)

# report these and verify they follow what we expect
print(np.mean(lns), np.var(lns)) # from the seqspec
print(minl, maxl) # from the reads
print(min_expected, max_expected) # from the fastqs

28.0 0.0
28 28
28 28


In [ ]:
# extract the barcodes and see what fraction of them are on the whitelist
# what about 1-ham? 2-ham?
